# Parcel trajectories inside a storm

This notebook draws the paths of air parcels through a thunderstorm. The
parcels come from CM1's Lagrangian parcel output. It covers:

1. choosing a set of parcels, here the ones in the storm's cloudy updraft;
2. reading their tracks from the parcel output;
3. drawing the tracks as *trails*: tubes behind each parcel's current
   position that narrow with age, with a cone at the head;
4. trails inside a see-through cloud, next to the physical cloud;
5. an animation in which the trails follow the parcels.

**How the trails are drawn.** `Scene.add_trajectories` takes a dataset of
tracks. For each time shown, skyvista keeps the part of every track that
lies within `trail_duration` seconds before that time. It writes those
points to the bundle. Blender rebuilds each trail as a curve and sweeps a
tube along it. No tube meshes are stored, so thousands of trails stay
cheap.

**What you need.** Run the notebook in the pixi `blender` environment of
the skyvista repository (`pixi run -e blender jupyter lab`). The path to
the CM1 run is in `notebook_setup.py`, next to this notebook. The run must
have parcel output (CM1's `iprcl = 1`). The renders use skyvista's
"preview" quality: 16 samples per pixel at 960 x 540.

In [ ]:
import sys
from pathlib import Path

# notebook_setup.py sits next to this notebook. Jupyter starts the kernel
# either in this folder or in the repository root (VS Code's default).
for notebook_folder in (Path.cwd(), Path.cwd() / "examples" / "notebooks"):
    if (notebook_folder / "notebook_setup.py").exists():
        sys.path.insert(0, str(notebook_folder))
        break

import time

import numpy as np
import xarray as xr
import xarrams

import notebook_setup as setup
import skyvista as sv
from skyvista.cloud_optics import cm1_morrison_hydrometeor_species

## 1. Choose the parcels

CM1 writes its parcels to one file per output time,
`cm1out_pdata_NNNNNN.nc`, numbered from 1 at t = 0. Each file holds every
parcel's position (`x`, `y`, `z`, in m) and its properties at that time.
The parcel index is the file's `xh` dimension, so a parcel keeps the
same index in every file.

This run has about 36 million parcels, far too many to draw. The notebook
therefore follows the parcels that are in the storm's cloudy updraft at
40 minutes. Two conditions define them:

- vertical velocity `w` above 5 m/s;
- cloud water mixing ratio `qc` above 0.01 g/kg.

Reading one variable for every parcel takes about a second. The files
are opened with `drop_variables=["xh"]` because xarray reads a dimension's
coordinate values eagerly, and here they are 36 million parcel numbers.

In [ ]:
# The time at which the parcels are chosen (s).
SELECTION_TIME_SECONDS = 2400.0
# The updraft: rising faster than this (m/s)...
UPDRAFT_MINIMUM_W = 5.0
# ...and inside the cloud: more cloud water than this (kg/kg).
CLOUD_MINIMUM_QC = 1.0e-5


def cm1_parcel_file(run_dir, time_seconds):
    """
    The CM1 parcel file nearest a model time.

    CM1 numbers its parcel files from 1 at t = 0, one per parcel output
    interval. The interval is the time of the second file.
    """
    output_dir = run_dir / "output"
    with xr.open_dataset(
        output_dir / "cm1out_pdata_000002.nc", engine="h5netcdf", drop_variables=["xh"]
    ) as second_file:
        output_interval = float(second_file["time"].values[0])
    file_number = int(round(time_seconds / output_interval)) + 1
    return output_dir / f"cm1out_pdata_{file_number:06d}.nc"


# w and qc of every parcel at the selection time.
selection_file = cm1_parcel_file(setup.CM1_ISOLATED_STORM_RUN, SELECTION_TIME_SECONDS)
with xr.open_dataset(selection_file, engine="h5netcdf", drop_variables=["xh"]) as parcels:
    parcel_w = parcels["w"].isel(time=0).values
    parcel_qc = parcels["qc"].isel(time=0).values

# Parcel indices (positions along xh) of the cloudy updraft.
in_cloudy_updraft = (parcel_w > UPDRAFT_MINIMUM_W) & (parcel_qc > CLOUD_MINIMUM_QC)
updraft_parcel_indices = np.flatnonzero(in_cloudy_updraft)
print(f"{parcel_w.size:,} parcels; {updraft_parcel_indices.size:,} in the cloudy updraft")

## 2. Read their tracks

skyvista expects tracks as an xarray `Dataset` with two dimensions:

- `trajectory_ix`: one entry per parcel;
- `time`: the output times.

The variables `x`, `y` and `z` hold the positions, in the same
coordinates as the gridded output (m). Other variables, such as `w`,
can color the trails.

The cell below reads the chosen parcels from every parcel file between 20
minutes before and 10 minutes after the selection time. It takes every
third file, one per 30 s: Blender draws the trails as smooth curves, so
the 10-s output adds little. Reading takes about 3 minutes. The tracks
are then saved under `WORK_DIR`, and later runs load that file instead.

The parcel files store packed integers. `drop_encoding()` removes that
on-disk format before saving, so the saved file holds plain floats.

In [ ]:
# The track times: 20 min before to 10 min after the selection (s).
TRACK_START_SECONDS = SELECTION_TIME_SECONDS - 20 * 60.0
TRACK_END_SECONDS = SELECTION_TIME_SECONDS + 10 * 60.0
TRACK_SPACING_SECONDS = 30.0
# Positions, plus vertical velocity for coloring.
TRACK_VARIABLES = ["x", "y", "z", "w"]

track_cache_path = setup.WORK_DIR / "updraft_parcel_tracks.nc"
if track_cache_path.exists():
    updraft_tracks = xr.load_dataset(track_cache_path)
else:
    track_times = np.arange(
        TRACK_START_SECONDS, TRACK_END_SECONDS + 1.0, TRACK_SPACING_SECONDS
    )
    read_start = time.perf_counter()
    # The chosen parcels at each time, one dataset per parcel file.
    tracks_by_time = []
    for track_time in track_times:
        parcel_path = cm1_parcel_file(setup.CM1_ISOLATED_STORM_RUN, track_time)
        with xr.open_dataset(parcel_path, engine="h5netcdf", drop_variables=["xh"]) as parcels:
            tracks_by_time.append(
                parcels[TRACK_VARIABLES].isel(time=0, xh=updraft_parcel_indices).load()
            )
    # Stack the times; label each track by its parcel index.
    updraft_tracks = (
        xr.concat(tracks_by_time, dim="time")
        .rename({"xh": "trajectory_ix"})
        .assign_coords(trajectory_ix=updraft_parcel_indices)
        .transpose("trajectory_ix", "time")
    )
    updraft_tracks.drop_encoding().to_netcdf(track_cache_path)
    print(f"read {len(track_times)} parcel files in {time.perf_counter() - read_start:.0f} s")

# Times as float64 seconds, the type of the gridded output's times.
updraft_tracks["time"] = updraft_tracks["time"].astype(float)
print(dict(updraft_tracks.sizes))

A few thousand trails crowd into one solid mass. The figures therefore
draw a random 600 of them. The storm's center is the median position of
the chosen parcels at the selection time; the camera aims at it.

In [ ]:
# The number of trails drawn, and the seed of their random choice.
TRAILS_DRAWN = 600
TRAIL_SAMPLE_SEED = 0

random_generator = np.random.default_rng(TRAIL_SAMPLE_SEED)
drawn_track_positions = np.sort(
    random_generator.choice(updraft_tracks.sizes["trajectory_ix"], TRAILS_DRAWN, replace=False)
)
drawn_tracks = updraft_tracks.isel(trajectory_ix=drawn_track_positions)

# The storm's center: where the chosen parcels are at the selection time.
parcels_at_selection = updraft_tracks.sel(time=SELECTION_TIME_SECONDS)
storm_center_x = float(parcels_at_selection["x"].median())
storm_center_y = float(parcels_at_selection["y"].median())
print(f"storm center at x = {storm_center_x / 1e3:.1f} km, y = {storm_center_y / 1e3:.1f} km")

## 3. Draw the trails

`add_trajectories` draws tubes by default (`style="tube"`). These
options shape the trails:

- `trail_duration`: the trail length in seconds. Each trail holds the
  parcel's positions from that long before the time shown up to it.
  Without it, a trail holds the whole history.
- `trail_taper`: the tube's radius at the trail's oldest end, as a
  fraction of its radius at the head. 0.1 narrows the trail to a tenth,
  so the direction of travel reads even without the cone.
- `tube_radius` (m) and `tube_resolution`, the number of vertices
  around the tube.
- `head_length_frac` and `head_radius_frac`: the head cone's length and
  base radius, in tube radii. `head_length_frac=0` removes the cone.
- `scalar`, `cmap` and `clim`: the coloring. Here `w` is colored on a
  diverging map, so rising air is red and sinking air blue.
- `limit`: the most tracks drawn. Its default is 1000, so `None` is
  needed to draw more.

`to_blender(..., times=[t])` draws the scene at the one time `t`. The
trails then end where the parcels are at 40 minutes. They show where
the air in the updraft came from over the previous 15 minutes.

In [ ]:
# Trails colored by w on a symmetric scale (m/s).
W_COLOR_LIMIT = 20.0
# A 15-minute trail for the stills (s).
STILL_TRAIL_SECONDS = 15 * 60.0


def add_parcel_trails(scene, trail_seconds):
    """
    Add the drawn parcels to ``scene`` as tapered trails colored by w.

    The trails are 30 m tubes that narrow to a tenth of that at their
    oldest end, with a cone at each head.
    """
    return scene.add_trajectories(
        drawn_tracks,
        scalar="w",
        cmap="RdYlBu_r",
        clim=(-W_COLOR_LIMIT, W_COLOR_LIMIT),
        limit=None,
        tube_radius=30.0,
        tube_resolution=6,
        head_length_frac=6.0,
        head_radius_frac=2.5,
        trail_duration=trail_seconds,
        trail_taper=0.1,
    )


# The camera: 26 km from the storm, to its south-south-east, 2 km up.
# The azimuth is counterclockwise from east.
CAMERA_DISTANCE = 26.0e3
CAMERA_AZIMUTH_FROM_STORM_DEG = -60.0
CAMERA_HEIGHT = 2000.0
# The aim: 7 km up, 2 km east of the center, toward the anvil.
AIM_HEIGHT, AIM_EAST_SHIFT = 7000.0, 2000.0


def make_storm_config():
    """
    Preview-quality export settings with the camera on the storm.

    The camera stands ``CAMERA_DISTANCE`` from the storm's center, in the
    direction ``CAMERA_AZIMUTH_FROM_STORM_DEG`` (counterclockwise from
    east). The figures below draw their own colorbars, so Blender's
    overlay colorbar is turned off.
    """
    config = sv.BlenderExportConfig.for_clouds(quality="preview")
    azimuth = np.deg2rad(CAMERA_AZIMUTH_FROM_STORM_DEG)
    config.camera.location = (
        storm_center_x + CAMERA_DISTANCE * np.cos(azimuth),
        storm_center_y + CAMERA_DISTANCE * np.sin(azimuth),
        CAMERA_HEIGHT,
    )
    config.camera.look_at = (storm_center_x + AIM_EAST_SHIFT, storm_center_y, AIM_HEIGHT)
    config.camera.lens_mm = 35.0
    # The sun 35 degrees up, in the west-south-west, to the camera's left.
    config.world.sun_azimuth_deg = -150.0
    config.world.sun_elevation_deg = 35.0
    config.overlay_colorbars = False
    return config

In [ ]:
trails_bundle_dir = setup.WORK_DIR / "trajectories_trails"
add_parcel_trails(sv.Scene(), STILL_TRAIL_SECONDS).to_blender(
    trails_bundle_dir,
    config=make_storm_config(),
    times=[SELECTION_TIME_SECONDS],
    render=True,
    quiet=True,
)
setup.show_image(trails_bundle_dir)

**What the trails show.** At 40 minutes the chosen parcels are at a
median height of 7.3 km. Their median vertical velocity there is 10 m/s.
Fifteen minutes earlier, their median height was 2.5 km. None was below
1.7 km then, so the trails start well above the ground. They only seem
to reach it because they meet the horizon in the image.

The pale lower parts of the trails are slow ascent between 2 and 3 km.
The red upper parts are the updraft core. There the fastest drawn parcel
rises at 29 m/s.

## 4. Trails inside the cloud

The cloud comes from CM1's gridded output, read as in the storm portrait
notebook (`01_storm_portrait.ipynb`, section 6). The crop keeps 50 x 50 km
around the storm and the levels up to 16.5 km.

In [ ]:
# Morrison mixing ratios (kg/kg), number concentrations (#/kg) and the
# air density (kg/m^3).
CM1_CLOUD_VARIABLES = ["qc", "qr", "qi", "qs", "qg", "nci", "ncr", "ncs", "ncg", "rho"]
# Half the width of the crop around the storm, and its top (m).
CROP_HALF_WIDTH, CROP_TOP = 25.0e3, 16.5e3
# Morrison settings of this run's namelist: ndcnst = 300 cm^-3, ihail = 1.
cm1_species = cm1_morrison_hydrometeor_species(
    droplet_number_concentration=300.0e6, graupel_density=900.0
)


def cm1_output_file(run_dir, time_seconds):
    """
    The CM1 gridded output file nearest a model time.

    CM1 numbers its files from 1 at t = 0, one per output interval. The
    interval is the time of the second file.
    """
    with xr.open_dataset(run_dir / "output" / "cm1out_000002.nc") as second_file:
        output_interval = float(second_file["time"].values[0])
    file_number = int(round(time_seconds / output_interval)) + 1
    return run_dir / "output" / f"cm1out_{file_number:06d}.nc"


def read_storm_clouds(times_seconds):
    """
    The hydrometeors around the storm at the given model times.

    The coordinates are renamed from CM1's ``xh``, ``yh``, ``zh`` to
    ``x``, ``y``, ``z`` (m). The result keeps its ``time`` dimension
    (s), one entry per time asked for.
    """
    output_paths = [
        cm1_output_file(setup.CM1_ISOLATED_STORM_RUN, time_seconds)
        for time_seconds in times_seconds
    ]
    cm1_output = xarrams.read_cm1_output(
        output_paths, keep_vars=CM1_CLOUD_VARIABLES, silent=True
    )
    return (
        cm1_output.rename({"xh": "x", "yh": "y", "zh": "z"})
        .sel(
            x=slice(storm_center_x - CROP_HALF_WIDTH, storm_center_x + CROP_HALF_WIDTH),
            y=slice(storm_center_y - CROP_HALF_WIDTH, storm_center_y + CROP_HALF_WIDTH),
            z=slice(0.0, CROP_TOP),
        )
        .load()
    )


storm_clouds = read_storm_clouds([SELECTION_TIME_SECONDS]).isel(time=0)
print("grid", dict(storm_clouds.sizes))

The physically based cloud hides the trails completely. Light gets only
tens of meters into the cloud, and the trails lie kilometers deep.

`extinction_scale` multiplies the cloud's extinction. A value of 0.03
cuts the extinction to 3% of its physical value. The cloud's outline stays
visible, and the trails show through it. The scene keeps the cloud's
physical lighting, so the thinned cloud still shades the trails.

In [ ]:
# The cloud as the eye would see it.
physical_bundle_dir = setup.WORK_DIR / "trajectories_cloud"
sv.Scene().add_clouds(storm_clouds, species=cm1_species).to_blender(
    physical_bundle_dir, config=make_storm_config(), render=True, quiet=True
)

# The thinned cloud with the trails inside it.
THINNED_EXTINCTION_SCALE = 0.03
thinned_bundle_dir = setup.WORK_DIR / "trajectories_in_cloud"
thinned_scene = sv.Scene().add_clouds(
    storm_clouds, species=cm1_species, extinction_scale=THINNED_EXTINCTION_SCALE
)
thinned_scene = add_parcel_trails(thinned_scene, STILL_TRAIL_SECONDS)
thinned_scene.to_blender(
    thinned_bundle_dir,
    config=make_storm_config(),
    times=[SELECTION_TIME_SECONDS],
    render=True,
    quiet=True,
)
print("rendered", physical_bundle_dir.name, "and", thinned_bundle_dir.name)

In [ ]:
# Text heights, as fractions of the panel height.
TITLE_SIZE, LABEL_SIZE = 0.04, 0.03
selection_minutes = SELECTION_TIME_SECONDS / 60.0

# The colorbar of the trails, as drawn by Blender.
w_colorbar = sv.compose.Colorbar(
    cmap="RdYlBu_r",
    clim=(-W_COLOR_LIMIT, W_COLOR_LIMIT),
    label="Parcel vertical velocity w (m/s)",
    ticks=list(np.linspace(-W_COLOR_LIMIT, W_COLOR_LIMIT, 5)),
    anchor="lower right",
    length=0.35,
    text_size=LABEL_SIZE,
)
left_panel = sv.compose.Panel(
    setup.rendered_frame(physical_bundle_dir),
    [
        sv.compose.Text("Cloud, physically based", size=TITLE_SIZE),
        sv.compose.Text(
            f"t = {selection_minutes:.0f} min", size=LABEL_SIZE, offset=(0.0, -0.055)
        ),
    ],
)
right_panel = sv.compose.Panel(
    setup.rendered_frame(thinned_bundle_dir),
    [
        sv.compose.Text("Updraft parcels, last 15 min", size=TITLE_SIZE),
        sv.compose.Text(
            f"Cloud extinction x {THINNED_EXTINCTION_SCALE}",
            anchor="lower left",
            size=LABEL_SIZE,
        ),
        w_colorbar,
    ],
)
figure_path = setup.WORK_DIR / "trajectories_side_by_side.png"
sv.compose.Figure(panels=[left_panel, right_panel]).render(figure_path)
setup.show_image(figure_path, width=1000)

**What the figure shows.** The physical cloud on the left hides every
trail. With its extinction cut to 3%, the cloud on the right lets the
trails show through the tower. The anvil stays visible as a pale veil,
because light crosses many kilometers of it. Most trails end below the
anvil, where the parcels are at 40 minutes.

## 5. An animation

An animation of 33 frames runs from 30 to 46 minutes, 30 s apart. At each
frame, each trail holds the last 5 minutes of its parcel's track, so the
trails move with the parcels. The trails are shorter than in the stills,
so each parcel's motion stays clear.

The cloud changes too, and 33 times of it would take 11 GB of memory.
`sv.BlenderAnimation` therefore builds the animation one frame at a time
(see `04_animation_and_tracking.ipynb`). Its `build_frame(time)` returns
the scene of one frame:

- the thinned cloud, read from that time's output file only;
- the trails, from the whole track dataset. skyvista cuts each trail to
  the frame's time.

The colors use the fixed limits `clim`, so a given w has the same color
in every frame. The frames are 640 x 360, to keep the render to a few
minutes.

In [ ]:
# Frame times (s), and the trail length in the animation (s).
ANIMATION_START_SECONDS, ANIMATION_END_SECONDS = 1800.0, 2760.0
FRAME_SPACING_SECONDS = 30.0
ANIMATION_TRAIL_SECONDS = 5 * 60.0
ANIMATION_RESOLUTION = (640, 360)
VIDEO_FPS = 8
# Frames exported in parallel; each holds one time of the cloud.
EXPORT_WORKERS = 4

frame_times = np.arange(
    ANIMATION_START_SECONDS, ANIMATION_END_SECONDS + 1.0, FRAME_SPACING_SECONDS
)


def build_trail_frame(frame_time):
    """
    One frame: the thinned cloud at ``frame_time`` and the trails up to it.

    Only this frame's cloud is read, so memory holds one time of it per
    export worker. The track dataset holds every time; the export cuts
    each trail to ``ANIMATION_TRAIL_SECONDS`` before ``frame_time``.
    """
    frame_clouds = read_storm_clouds([frame_time]).isel(time=0)
    frame_scene = sv.Scene().add_clouds(
        frame_clouds, species=cm1_species, extinction_scale=THINNED_EXTINCTION_SCALE
    )
    return add_parcel_trails(frame_scene, ANIMATION_TRAIL_SECONDS)


animation_config = make_storm_config()
animation_config.render.resolution = ANIMATION_RESOLUTION

Each rendered frame gets a title, its model time and the colorbar.
`TimeStamp` reads the frame's time from the animation, and its
`formatter` turns the seconds into text. `run` then exports, assembles,
renders, composes and encodes the video.

In [ ]:
animation_figure = sv.compose.Figure(
    panels=[
        sv.compose.Panel(
            lambda frame: frame.render_path,
            [
                sv.compose.Text("Updraft parcels, last 5 min", size=TITLE_SIZE),
                sv.compose.TimeStamp(
                    formatter=lambda seconds: f"t = {seconds / 60:.1f} min",
                    size=LABEL_SIZE,
                    offset=(0.0, -0.055),
                ),
                w_colorbar,
            ],
        )
    ]
)

trail_animation = sv.BlenderAnimation(
    setup.WORK_DIR / "trajectories_animation",
    times=list(frame_times),
    build_frame=build_trail_frame,
    config=animation_config,
    fps=VIDEO_FPS,
)
run_start = time.perf_counter()
trajectory_video = trail_animation.run(figure=animation_figure, export_workers=EXPORT_WORKERS)
print(f"animation made in {(time.perf_counter() - run_start) / 60:.1f} min")
setup.show_video(trajectory_video)

Six of the frames, for viewers where the video does not play:

In [ ]:
# Every sixth composed frame, 3 x 2.
CONTACT_SHEET_FRAME_STEP = 6
composed_frame_paths = sorted(trail_animation.composed_dir.glob("frame_*.png"))
contact_sheet_path = setup.WORK_DIR / "trajectories_animation_sheet.png"
sv.compose.Figure(
    panels=[
        sv.compose.Panel(path)
        for path in composed_frame_paths[::CONTACT_SHEET_FRAME_STEP]
    ],
    n_columns=3,
).render(contact_sheet_path)
setup.show_image(contact_sheet_path, width=1000)

**What the animation shows.** At 30 minutes the parcels rise slowly in
the lower part of the tower. Their trails are therefore short and pale.
From about 36 minutes they accelerate into the red column of the updraft
core. By the last frame, at 46 minutes, two thirds have passed 9 km and
slowed in the anvil.

The tracks continue past the animation. At 50 minutes, about three
quarters of the chosen parcels are above 9 km. Their median vertical
velocity is then close to zero.

## Where to go from here

- **Particles.** `style="particle"` draws each parcel as a sphere of
  `particle_radius` at its current position, with no trail. Blender
  renders them as a native point cloud, so 10^5-10^6 particles stay
  cheap. A particle spec and a trail spec can share one scene.
- **Baked tubes.** `blender_instancing=False` exports PyVista's tube
  meshes instead of curves. The trails then do not taper.
- **Other track data.** Any dataset with `trajectory_ix` and `time`
  dimensions and `x`, `y`, `z` variables works, for example tracked
  thermals or trajectories computed after the model run.
- **PyVista.** The same `add_trajectories` call works with
  `Scene.show()` and `Scene.animate()`. `trail_duration` applies there
  too; the PyVista tubes stay uniform.